In [1]:
import numpy as np
import pandas as pd
import os
from faker import Faker

In [2]:
def generate_loan_data(n=227000,seed = 42):
    """ Generate synthetic loan dataset """
    fake=Faker('en_IN')
    np.random.seed(seed)

    print(f'Generating {n} synthetic loan dataset.....')

    # 1. Generate core independent features
    age = np.random.randint(21, 65, n)
    annual_income_lakhs = np.round(np.random.lognormal(2.5, 0.7, n).clip(1.5, 100), 2)

    # 2. FIX: Make Credit History dependent on Age (A 21yo can't have 25 years of history)
    # Max possible credit history is roughly (Age - 18)
    max_possible_history = np.clip(age - 18, 0, 25)
    credit_history_years = np.array([np.random.randint(0, max(1, m)) for m in max_possible_history])

    # 3. FIX: Replace Uniform Credit Score with a Realistic Normal Distribution
    # Most scores cluster around 700; higher credit history slightly nudges the score up
    base_score = np.random.normal(loc=680, scale=95, size=n)
    history_boost = credit_history_years * 2  # More history slightly boosts score
    credit_score = np.clip(base_score + history_boost, 300, 900).astype(int)

    # 4. FIX: Make Loan Amount and EMIs logically scale with Income
    # Loan amount is capped/correlated to income to avoid impossible ratios
    loan_amount_lakhs = np.round((annual_income_lakhs * np.random.uniform(0.5, 4.0, n)).clip(0.5, 200), 2)

    # Existing loans determines existing EMI
    existing_loans = np.random.choice([0, 1, 2, 3, 4], n, p=[0.35, 0.30, 0.20, 0.10, 0.05])
    # If they have loans, calculate an EMI relative to income, otherwise 0
    monthly_emi_existing = np.where(
        existing_loans > 0,
        np.round((annual_income_lakhs * 100000 / 12) * np.random.uniform(0.1, 0.4, n), 0),
        0
    )

    # 5. Assemble everything into your target structure
    data = {
        'application_id': [f'LN{fake.unique.random_number(digits=8)}' for _ in range(n)],
        'age': age,
        'gender': np.random.choice(['M', 'F'], n, p=[0.65, 0.35]),
        'marital_status': np.random.choice(['single', 'married', 'divorced'], n, p=[0.3, 0.6, 0.1]),
        'dependents': np.random.choice([0, 1, 2, 3, 4], n, p=[0.25, 0.3, 0.25, 0.15, 0.05]),
        'education': np.random.choice(['graduate', 'post_graduate', 'under_graduate'], n, p=[0.45, 0.25, 0.30]),
        'employment_type': np.random.choice(['salaried', 'self_employed', 'business'], n, p=[0.55, 0.25, 0.20]),
        'annual_income_lakhs': annual_income_lakhs,
        'loan_amount_lakhs': loan_amount_lakhs,
        'loan_tenure_months': np.random.choice([12, 24, 36, 48, 60, 84, 120, 180, 240], n),
        'interest_rate': np.round(np.random.uniform(7.5, 16.5, n), 2),
        'credit_score': credit_score,
        'existing_loans': existing_loans,
        'credit_history_years': credit_history_years,
        'monthly_emi_existing': monthly_emi_existing,
        'property_area': np.random.choice(['urban', 'semi-urban', 'rural'], n, p=[0.4, 0.35, 0.25]),
        'has_co_applicant': np.random.choice([0, 1], n, p=[0.6, 0.4]),
    }

    df = pd.DataFrame(data)

    # Derived Features
    df['debt_to_income']=(df['monthly_emi_existing']*12)/(df['annual_income_lakhs']*100000)
    df['loan_to_income'] = df['loan_amount_lakhs']/df['annual_income_lakhs']

    # Target: loan default (~15% realistic default rate cascade)
    
    # 1. Establish a strong baseline risk based entirely on the Credit Score
    # Lower scores get a much higher base probability of default
    base_prob = np.where(df['credit_score'] < 500, 0.60,
                np.where(df['credit_score'] < 600, 0.30,
                np.where(df['credit_score'] < 700, 0.10,
                np.where(df['credit_score'] < 800, 0.02, 0.001))))

    # 2. Layer on additive risk factors (high debt, high loan-to-income, etc.)
    # Instead of an arbitrary score threshold, these directly increase the probability of default
    risk_multipliers = (
        (df['debt_to_income'] > 0.5).astype(int) * 0.15 +
        (df['loan_to_income'] > 5).astype(int) * 0.15 +
        (df['existing_loans'] >= 3).astype(int) * 0.10 +
        (df['credit_history_years'] < 2).astype(int) * 0.08 +
        (df['age'] < 25).astype(int) * 0.05
    )
    
    # Combined base probability and risk flags
    total_prob = base_prob + risk_multipliers
    
    # 3. Add a smaller, realistic amount of random noise to the probability
    # Scale of 0.03 means noise gently shifts the probability by +/- 6% at most
    total_prob += np.random.normal(0, 0.03, n)
    
    # Keep probabilities strictly bounded between 0% and 100%
    total_prob = np.clip(total_prob, 0.0, 1.0)
    
    # 4. Generate the final binary default variable using a Binomial distribution
    # This flips a weighted coin for each row based on its custom calculated probability
    df['default'] = np.random.binomial(1, total_prob)

    # Missing value (~3.5%)
    for col in ['annual_income_lakhs','credit_score','credit_history_years']:
        mask = np.random.random(n) < 0.035
        df.loc[mask,col] = np.nan

    return df
    

In [3]:
def save_data(df,output_dir='data/raw'):
    os.makedirs(output_dir,exist_ok=True)
    filepath=os.path.join(output_dir,'loan_applications.csv')
    df.to_csv(filepath,index=False)
    print(f'Saved : {filepath} ({len(df)} records) , {df['default'].mean():.1%} default rate')
    return filepath

In [4]:
df=generate_loan_data()

Generating 227000 synthetic loan dataset.....


In [5]:
save_data(df)

Saved : data/raw\loan_applications.csv (227000 records) , 12.8% default rate


'data/raw\\loan_applications.csv'